In [1]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import warnings
import os
from pathlib import Path

import joblib
import pandas as pd
from sklearn.base import clone
import sys
sys.path.append('/content')
from utils.helper import SMEFeatureEngineer
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import (
    AdaBoostClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv("Data\SMEs_Data.csv")
df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')
    


In [4]:
MODELS_AND_GRIDS = {
    'Gradient Boosting': (
        GradientBoostingClassifier(random_state=42),
        {
            'classifier__n_estimators': [100, 200, 300],
            'classifier__learning_rate': [0.03, 0.08, 0.15],
            'classifier__max_depth': [3, 4, 5],
            'classifier__min_samples_leaf': [3, 5, 10],
            'classifier__subsample': [0.7, 0.85, 1.0],
        },
    ),
    'XGBoost': (
        XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=-1),
        {
            'classifier__n_estimators': [100, 200, 300],
            'classifier__learning_rate': [0.03, 0.08, 0.15],
            'classifier__max_depth': [3, 5, 7],
            'classifier__subsample': [0.7, 0.85, 1.0],
            'classifier__colsample_bytree': [0.7, 0.85, 1.0],
        },
    ),
    'Hist Gradient Boosting': (
        HistGradientBoostingClassifier(class_weight='balanced', random_state=42),
        {
            'classifier__max_iter': [100, 200, 300],
            'classifier__learning_rate': [0.03, 0.08, 0.15],
            'classifier__max_depth': [3, 5, 7, None],
            'classifier__min_samples_leaf': [10, 20, 30],
        },
    ),
    'Random Forest': (
        RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
        {
            'classifier__n_estimators': [100, 200, 300],
            'classifier__max_depth': [8, 12, 16, None],
            'classifier__min_samples_leaf': [2, 4, 6],
            'classifier__max_features': ['sqrt', 'log2'],
        },
    ),
    'Extra Trees': (
        ExtraTreesClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
        {
            'classifier__n_estimators': [100, 200, 300],
            'classifier__max_depth': [10, 14, 18, None],
            'classifier__min_samples_leaf': [2, 3, 5],
            'classifier__max_features': ['sqrt', 'log2'],
        },
    ),
    'AdaBoost': (
        AdaBoostClassifier(random_state=42),
        {
            'classifier__n_estimators': [50, 100, 150, 250],
            'classifier__learning_rate': [0.05, 0.1, 0.5, 1.0],
        },
    ),
    'Logistic Regression': (
        LogisticRegression(
            class_weight='balanced', solver='lbfgs', max_iter=1000, random_state=42
        ),
        {
            'classifier__C': [0.01, 0.1, 0.5, 1.0, 5.0],
        },
    ),
}

In [5]:
def main():
    print('=' * 60)
    print('   SME Risk Intelligence — Hyperparameter Optimization Only')
    print('=' * 60)

    import os
    
    
    df = pd.read_csv("Data\SMEs_Data.csv")
    print(f"\n[OK] Data loaded  : {df.shape[0]:,} rows × {df.shape[1]} columns")

    TARGET = 'risk_sharp'
    FEATURES = [c for c in df.columns if c != TARGET]

    X = df[FEATURES]
    y = df[TARGET]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=42
    )
    print(f"\n[OK] Train: {len(X_train):,} | Test: {len(X_test):,}")

    preprocessor = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler()),
    ])

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    all_results = {}

    print('\n' + '=' * 60)
    print('   STARTING TUNING PROCESS')
    print('=' * 60)

    for name, (model, param_grid) in MODELS_AND_GRIDS.items():
        iterations = 15 if name != 'Logistic Regression' and name != 'AdaBoost' else 5
        print(f"\n[RUNNING] Optimizing {name}...")

        pipe = Pipeline([
            ('feat_eng', SMEFeatureEngineer()),
            ('preprocessor', preprocessor),
            ('classifier', model),
        ])

        grid = RandomizedSearchCV(
            estimator=pipe,
            param_distributions=param_grid,
            n_iter=iterations,
            scoring='roc_auc',
            cv=cv,
            n_jobs=-1,
            refit=False, # شيلنا الـ refit عشان مياخدش وقت زيادة في إعادة تدريب الموديل بالكامل
            random_state=42,
            verbose=0,
        )
        grid.fit(X_train, y_train)

        # تنظيف الـ parameters من كلمة 'classifier__' عشان تاخدها جاهزة للموديل فوراً
        clean_params = {k.replace('classifier__', ''): v for k, v in grid.best_params_.items()}

        all_results[name] = {
            'best_params': clean_params,
            'best_cv_roc_auc': round(grid.best_score_ * 100, 3),
        }

        print(f"   [DONE] Best CV ROC-AUC: {grid.best_score_ * 100:.3f}%")
        print(f"   [PARAMS] {clean_params}")

    print('\n' + '=' * 70)
    print('   FINAL REPORT: BEST HYPERPARAMETERS PER MODEL')
    print('=' * 70)
    
    for name, result in all_results.items():
        print(f"\n★ Model: {name}")
        print(f"  -> Best CV ROC-AUC: {result['best_cv_roc_auc']}%")
        print(f"  -> Copy these Parameters:")
        print(json.dumps(result['best_params'], indent=4))
        print('-' * 50)

    print('\n' + '=' * 70)
    print('   OPTIMIZATION COMPLETE — COPY YOUR PARAMS ABOVE')
    print('=' * 70)

if __name__ == '__main__':
    main()

   SME Risk Intelligence — Hyperparameter Optimization Only

[OK] Data loaded  : 35,000 rows × 12 columns

[OK] Train: 28,000 | Test: 7,000

   STARTING TUNING PROCESS

[RUNNING] Optimizing Gradient Boosting...
   [DONE] Best CV ROC-AUC: 97.983%
   [PARAMS] {'subsample': 1.0, 'n_estimators': 300, 'min_samples_leaf': 3, 'max_depth': 3, 'learning_rate': 0.15}

[RUNNING] Optimizing XGBoost...
   [DONE] Best CV ROC-AUC: 97.938%
   [PARAMS] {'subsample': 1.0, 'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.15, 'colsample_bytree': 1.0}

[RUNNING] Optimizing Hist Gradient Boosting...
   [DONE] Best CV ROC-AUC: 97.972%
   [PARAMS] {'min_samples_leaf': 30, 'max_iter': 300, 'max_depth': 3, 'learning_rate': 0.15}

[RUNNING] Optimizing Random Forest...
   [DONE] Best CV ROC-AUC: 97.413%
   [PARAMS] {'n_estimators': 300, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 16}

[RUNNING] Optimizing Extra Trees...
   [DONE] Best CV ROC-AUC: 97.153%
   [PARAMS] {'n_estimators': 100, 'm